# BC Parks — Photos & Geospatial

Second exploratory notebook. Combines the structured catalog with:

- **`data/boundaries.geojson`** — 930 park polygons from BC Open Data (Tantalis)
- **`data/photos/`** — 1,707 downloaded park photos

Run these from the project root first if any file is missing:

```bash
python -m src.download              # parks, advisories, activities, facilities
python -m src.download_boundaries   # park polygons
python -m src.download_photos       # 1.0 GB of images
```

Sections:
1. Load parks + boundaries + photos
2. Choropleth — photos per park
3. Choropleth — currently active advisories per park
4. Choropleth — open activities per park
5. Photo gallery (sanity check that images decode)
6. Area sanity-check: API `totalArea` vs Tantalis `OFFICIAL_AREA_HA`

In [ ]:
import json
import random
from pathlib import Path

import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from PIL import Image

DATA = Path('..') / 'data'

parks      = pd.json_normalize(json.loads((DATA / 'parks.json').read_text()))
advisories = pd.json_normalize(json.loads((DATA / 'advisories.json').read_text()))
activities = pd.json_normalize(json.loads((DATA / 'activities.json').read_text()))
photos     = pd.json_normalize(json.loads((DATA / 'photos.json').read_text()))
boundaries = json.loads((DATA / 'boundaries.geojson').read_text())

for f in boundaries['features']:
    f['id'] = f['properties'].get('ORCS_PRIMARY')

print(f'parks      : {len(parks)}')
print(f'advisories : {len(advisories)}')
print(f'activities : {len(activities)}')
print(f'photos     : {len(photos)}')
print(f'polygons   : {len(boundaries["features"])}')

## 1. Photos per park — choropleth

Color each park polygon by the number of photos in our local `data/photos/` directory.

In [ ]:
photos_per_park = (
    photos.groupby('orcs').size().reset_index(name='photo_count')
)
photos_per_park['orcs'] = photos_per_park['orcs'].astype('Int64')

valid_orcs = {f['id'] for f in boundaries['features']}
df = photos_per_park[photos_per_park['orcs'].isin(valid_orcs)].copy()
df['orcs_str'] = df['orcs'].astype(str)

fig = px.choropleth_mapbox(
    df,
    geojson=boundaries,
    locations='orcs_str',
    color='photo_count',
    color_continuous_scale='Viridis',
    range_color=(0, df['photo_count'].quantile(0.95)),
    mapbox_style='open-street-map',
    center={'lat': 54.5, 'lon': -125.0},
    zoom=3.7,
    opacity=0.7,
    title=f'Photos per park (n={df["photo_count"].sum()} photos across {len(df)} parks)',
)
fig.update_layout(height=650, margin={'r': 0, 't': 50, 'l': 0, 'b': 0})
fig.show()

## 2. Currently-active advisories per park

In [ ]:
now = pd.Timestamp.now(tz='UTC')
adv = advisories.copy()
for c in ['effectiveDate', 'endDate']:
    adv[c] = pd.to_datetime(adv[c], errors='coerce', utc=True)
active = adv[
    adv['advisoryStatus.advisoryStatus'].eq('Published')
    & adv['effectiveDate'].le(now)
    & (adv['endDate'].isna() | adv['endDate'].ge(now))
]

rows = []
for nodes in active['protectedAreas_connection.nodes']:
    if isinstance(nodes, list):
        for n in nodes:
            rows.append(n.get('orcs'))
adv_per_park = (
    pd.Series(rows, name='orcs').dropna().astype('Int64')
    .value_counts().rename_axis('orcs').reset_index(name='active_advisories')
)

df = adv_per_park[adv_per_park['orcs'].isin(valid_orcs)].copy()
df['orcs_str'] = df['orcs'].astype(str)

fig = px.choropleth_mapbox(
    df,
    geojson=boundaries,
    locations='orcs_str',
    color='active_advisories',
    color_continuous_scale='Reds',
    mapbox_style='open-street-map',
    center={'lat': 54.5, 'lon': -125.0},
    zoom=3.7,
    opacity=0.8,
    title=f'Active advisories per park (total events: {df["active_advisories"].sum()})',
)
fig.update_layout(height=650, margin={'r': 0, 't': 50, 'l': 0, 'b': 0})
fig.show()

## 3. Open activities per park

In [ ]:
act = activities[
    activities['isActive'].fillna(False) & activities['isActivityOpen'].fillna(False)
]
act_per_park = (
    act.groupby('protectedArea.orcs').size()
    .rename_axis('orcs').reset_index(name='open_activities')
)
act_per_park['orcs'] = act_per_park['orcs'].astype('Int64')

df = act_per_park[act_per_park['orcs'].isin(valid_orcs)].copy()
df['orcs_str'] = df['orcs'].astype(str)

fig = px.choropleth_mapbox(
    df,
    geojson=boundaries,
    locations='orcs_str',
    color='open_activities',
    color_continuous_scale='Greens',
    mapbox_style='open-street-map',
    center={'lat': 54.5, 'lon': -125.0},
    zoom=3.7,
    opacity=0.7,
    title='Open activities per park',
)
fig.update_layout(height=650, margin={'r': 0, 't': 50, 'l': 0, 'b': 0})
fig.show()

## 4. Photo gallery sanity check

Sample 16 random photos from `data/photos/` and decode them. If any of these fail to render, the manifest's `bytes` column is the place to look.

In [ ]:
manifest = pd.read_csv(DATA / 'photos_manifest.csv')
ok = manifest[manifest['status'].isin(['ok', 'skip']) & manifest['path'].notna()].copy()
ok = ok.merge(
    photos[['documentId', 'title', 'caption', 'orcs']],
    on='documentId', how='left',
)
name_by_orcs = parks.set_index('orcs')['protectedAreaName'].to_dict()
ok['parkName'] = ok['orcs_x'].map(name_by_orcs)

random.seed(42)
sample = ok.sample(n=16, random_state=42)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    img_path = Path('..') / r['path']
    try:
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
    except Exception as e:
        ax.text(0.5, 0.5, str(e), ha='center', va='center')
    title = (r['parkName'] or '')[:30]
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Area sanity check

Compare the API's `totalArea` (hectares) against Tantalis's `OFFICIAL_AREA_HA`. Big discrepancies tell you which parks have boundary data quality issues.

In [ ]:
b = pd.read_csv(DATA / 'boundaries.csv')
b = b.dropna(subset=['orcs'])
b = b.groupby('orcs', as_index=False)['OFFICIAL_AREA_HA'].sum()

cmp = parks[['orcs', 'protectedAreaName', 'totalArea']].merge(
    b, on='orcs', how='inner'
)
cmp = cmp[cmp['totalArea'].notna() & (cmp['totalArea'] > 0)].copy()
cmp['delta_pct'] = (cmp['OFFICIAL_AREA_HA'] - cmp['totalArea']) / cmp['totalArea'] * 100

print('summary of |delta %|:')
print(cmp['delta_pct'].abs().describe())

print('\nTop 10 mismatches:')
print(
    cmp.reindex(cmp['delta_pct'].abs().sort_values(ascending=False).index)
    [['protectedAreaName', 'totalArea', 'OFFICIAL_AREA_HA', 'delta_pct']].head(10).to_string(index=False)
)

fig = px.scatter(
    cmp, x='totalArea', y='OFFICIAL_AREA_HA',
    log_x=True, log_y=True,
    hover_name='protectedAreaName',
    title='API totalArea vs Tantalis OFFICIAL_AREA_HA (log-log; diagonal = match)',
)
fig.add_shape(type='line', x0=1, y0=1, x1=1e7, y1=1e7,
              line={'dash': 'dash', 'color': 'gray'})
fig.update_layout(height=550)
fig.show()